# Main Classes in the Benchmark Pipeline

This notebook explains the repository's real `BaseModel`, `BaseDataset`, `BaseBenchmark`, `ModelRun`, and `BenchmarkRun` classes. Every named example below exists in this codebase. Executable cells inspect local source files only: they do not load model weights, contact an API, or download a dataset.

In [ ]:
from pathlib import Path
import ast

ROOT = Path.cwd()

def source_node(relative_path: str, name: str) -> str:
    """Return a real class or function definition from this repository."""
    path = ROOT / relative_path
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)
    for node in ast.walk(tree):
        if isinstance(node, (ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            lines = source.splitlines()
            return "\n".join(lines[node.lineno - 1:node.end_lineno])
    raise KeyError(f"{name!r} not found in {relative_path}")

print("Reading source from:", ROOT)

## Architecture at a glance

| Concept | Real class | Responsibility |
| --- | --- | --- |
| Model | `models._base_model.BaseModel` | Given an image and a prompt, return generated text. |
| Dataset | `dataset._base_dataset.BaseDataset` | Supply rows, images, and ground-truth information. |
| Benchmark | `benchmarks._base_benchmark.BaseBenchmark` | Turn rows into evaluation tasks, call the model, and score the outputs. |
| Model run | `runners.model_run.ModelRun` | Hold one loaded model under a result-file name and optionally store load time. |
| Benchmark run | `runners.benchmark_run.BenchmarkRun` | Hold one benchmark and the number of samples to evaluate. |

### Precisely what flows between the methods?

Yes, rows from `dataset.get_samples()` are used by the benchmark, but a whole row is **not** passed directly into `model.predict()`:

```text
dataset.get_samples(...) -> rows

for each row:
    benchmark.get_image_for_row(row)       -> image
    benchmark.make_prompt(..., row, image) -> prompt string
    model.predict(image, prompt)           -> prediction string
    benchmark.evaluate_prediction(row, prediction, image) -> correctness/metrics
```

The row contains source data and expected answers. The benchmark extracts the model's visual input and constructs its textual instruction from that row. The model receives only the prepared `image` and `prompt`; the benchmark keeps the row so it can compare the returned prediction with ground truth.

A **run matrix** is just every selected model paired with every selected benchmark. For two models and three benchmarks, `runners.execution.run_benchmark_matrix()` performs six independent runs and writes six JSON result files.

In [ ]:
# This is the real method that implements the per-row flow described above.
print(source_node("benchmarks/_base_benchmark.py", "run"))

## 1. Models

`BaseModel` requires `predict(image, prompt) -> str`. The benchmark does not need to know whether generation occurs in local model weights or through an API, because both implementations expose the same method.

### Local model and remote API examples actually present here

- `Llava`, `Gemma`, `Falcon`, and `Qwen25VL3B` / `Qwen25VL72B` run Transformers model generation locally after their weights and processors have been loaded.
- `GPT4`, `GPT41`, and the GPT-5 wrappers inherit `OpenAIResponsesVisionModel`, which implements the same contract using OpenAI's Responses API: it converts the `PIL.Image` into a JPEG data URL and sends text plus image to the remote model.

### Preprocessing and post-processing

It is accurate to say that **preprocessing converts or adapts inputs into the format a model can properly read**. It can include more than changing a file type: for these models it also includes formatting a chat prompt, tokenizing text, converting pixels to tensors, batching inputs, and moving tensors to the model device.

`Llava.predict()` is a concrete example:

```python
prepared_prompt = self._prepare_prompt(prompt)
inputs = self.processor(text=prepared_prompt, images=image, return_tensors="pt").to(self.model.device)
output = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens, do_sample=False)
output_text = self.processor.decode(generated, skip_special_tokens=True)
return self._clean_output_text(output_text, prompt=prompt)
```

Yes: **output decoding is a form of post-processing**. `decode(...)` changes generated token IDs into readable text. In `Llava`, `_clean_output_text(...)` is a further post-processing step that constrains that text for benchmark formats such as one label or bounding-box lines.

In [ ]:
print(source_node("models/_base_model.py", "BaseModel"))
print("\n--- Llava.predict (local Transformers inference) ---\n")
print(source_node("models/llava15_7b.py", "predict"))
print("\n--- OpenAIResponsesVisionModel.predict (remote API inference) ---\n")
print(source_node("models/_openai_vision.py", "predict"))

### Do we use testing stubs?

Yes. They are used in the automated tests, not as production model implementations. In `tests/test_benchmark_runner.py`, `_StubModel` stores `name` and `max_new_tokens`, and `_StubBenchmark.run(...)` returns a small deterministic report. This lets tests verify matrix execution, output paths, sample counts, and model-load statistics without loading large models or datasets.

In [ ]:
print(source_node("tests/test_benchmark_runner.py", "_StubModel"))
print("\n--- Test-only benchmark stub ---\n")
print(source_node("tests/test_benchmark_runner.py", "_StubBenchmark"))

### Model inheritance tree

The tree groups checkpoint-only leaf classes on one line; each grouped class is separately exported from `models/__init__.py`.

```text
BaseModel
+-- OpenAIResponsesVisionModel
|   +-- GPT4 (gpt-4o), GPT41
|   `-- GPT5VisionModel
|       `-- GPT5, GPT51, GPT52, GPT53ChatLatest, GPT54, GPT54Mini,
|           GPT54Nano, GPT55, O3
`-- HuggingFaceModelBase  (also inherits CacheFirstLoaderMixin)
    +-- AutoProcessorModelBase
    |   +-- Llava
    |   |   +-- Llava15_13B
    |   |   `-- LlavaGemma2B
    |   +-- Gemma
    |   |   `-- PaliGemma3BMix448, PaliGemma2_3BMix448, PaliGemma2_10BMix448
    |   +-- _Qwen25VLBase
    |   |   `-- Qwen25VL3B, Qwen25VL7B, Qwen25VL32B, Qwen25VL72B
    |   `-- AutoImageTextToTextModelBase
    |       `-- LlavaOnevision15_4BInstruct, Qwen3VL4B, Qwen3VL8B,
    |           Qwen35_4B, Qwen35_9B, Gemma3_4B, Gemma3_12B, Gemma3_27B,
    |           Gemma4E2B, Gemma4E4B, Gemma4_26BA4B, Gemma4_31B,
    |           MiniCPMV46, MiniCPMV46Thinking
    +-- LlavaNextModelBase
    |   `-- Falcon, LlavaNextMistral7B, LlavaNextVicuna13B, Llama3LlavaNext8B
    +-- LlavaOnevision
    |   `-- LlavaOnevisionQwen2_7B
    +-- InternVL25
    |   `-- InternVL25_4B, InternVL3_2B, InternVL3_8B, InternVL35_8BInstruct
    `-- MiniCPMV26
        `-- MiniCPMo26
```

The shared parents prevent checkpoint wrappers from repeating model-loading and inference logic. For example, all Qwen2.5-VL leaf classes inherit `_Qwen25VLBase.predict()`, while current Transformers chat-template families such as Qwen3.5, Gemma 4, and MiniCPM-V 4.6 inherit `AutoImageTextToTextModelBase.predict()`.

`OpenAIResponsesVisionModel` centralizes image-plus-text requests to the Responses API. `GPT5VisionModel` changes only request compatibility behavior for GPT-5-family models by not unconditionally sending `temperature`. `HuggingFaceModelBase`, `AutoProcessorModelBase`, and `AutoImageTextToTextModelBase` live in `_hf_model.py`; `LlavaNextModelBase` lives in `_llava_next.py`.

## 2. Datasets

`BaseDataset` defines the minimum data-access interface. It does not decide whether a prediction is correct. For example:

- `ImageNet1k.get_samples(n)` selects image rows, `get_image_from_row(row)` converts an image to RGB, and `get_labels_img(row)` obtains acceptable class labels for that row.
- `GQA.get_samples(n)` joins image data with question/answer instructions; `get_question_from_row(row)` and `get_answers_from_row(row)` expose QA ground truth.
- `MSCOCOCaption.get_captions_from_row(row)` exposes reference captions; its benchmark decides to score captions with a BLEU-style metric.

In [ ]:
print(source_node("dataset/_base_dataset.py", "BaseDataset"))
print("\n--- Real QA dataset example: GQA.get_samples ---\n")
print(source_node("dataset/gqa.py", "get_samples"))
print("\n--- Real caption dataset accessor: MSCOCOCaption.get_captions_from_row ---\n")
print(source_node("dataset/mscoco_caption.py", "get_captions_from_row"))

### Dataset inheritance tree

```text
BaseDataset
+-- GQA                  +-- Flickr30k           +-- Flickr30kEntities
+-- ImageNet1k           +-- INaturalist         +-- MSCOCO
+-- MSCOCOCaption        +-- UCF101
`-- HFBaseDataset
    +-- HFClassificationDataset
    |   +-- FairFace       +-- FashionMNIST  +-- LSUN
    |   +-- MVTecAD        +-- OpenImagesV4  +-- Places
    |   `-- LVIS
    +-- HFQADataset
    |   +-- DocVQA         +-- VQAv2        +-- VisualCoT
    |   `-- VisualGenome
    +-- HFCaptionDataset
    |   `-- TextCaps
    +-- HFVideoClassificationDataset
    |   +-- DFDC           `-- Kinetics
    `-- HFMultipleChoiceSourceDataset
        +-- BLIP3o60k     +-- ConceptualCaptions  +-- InternVid
        +-- LAION400M     +-- LAION5B             `-- OpenVid1M
```

## 3. Benchmarks

A benchmark defines the evaluated task, independently from how a model generates text and how an upstream dataset stores records. Its key responsibilities are prompt construction, scoring, and report assembly.

| Existing benchmark family | Output requested from model | Existing scoring behavior |
| --- | --- | --- |
| `ClassificationBenchmark` | One label | Normalized label comparison inherited from `BaseBenchmark`. |
| `VisualQABenchmark` | Answer text | Normalized exact match against accepted answers. |
| `CaptioningBenchmark` | One caption | BLEU-style score against reference captions. |
| `DetectionBenchmark` | Labeled bounding boxes | IoU matching with precision, recall, and F1. |
| `MultipleChoiceBenchmark` | Choice letter or text | Match selected choice to the answer. |
| `VideoClassificationBenchmark` | One label for a frame sheet | Normalized label comparison. |

Concrete benchmark classes connect a dataset to one of these task definitions. `GQABenchmark`, for example, identifies `GQA` as its `dataset_cls` and inherits QA prompt/scoring behavior from `VisualQABenchmark`. `MSCOCOCaptionBenchmark` binds `MSCOCOCaption` to the captioning behavior; `MSCOCOBenchmark` binds `MSCOCO` to detection behavior.

In [ ]:
print(source_node("benchmarks/visual_qa/_visual_qa.py", "VisualQABenchmark"))
print("\n--- Thin concrete binding: GQABenchmark ---\n")
print(source_node("benchmarks/visual_qa/gqa.py", "GQABenchmark"))
print("\n--- Thin concrete binding: MSCOCOCaptionBenchmark ---\n")
print(source_node("benchmarks/captioning/mscoco_caption.py", "MSCOCOCaptionBenchmark"))

### Benchmark inheritance tree

```text
BaseBenchmark
+-- ClassificationBenchmark
|   +-- FairFaceBenchmark       +-- FashionMNISTBenchmark
|   +-- ImageNet1kBenchmark     +-- INaturalistBenchmark
|   +-- LSUNBenchmark           +-- MVTecADBenchmark
|   +-- OpenImagesV4Benchmark   `-- PlacesBenchmark
+-- CaptioningBenchmark
|   +-- ConceptualCaptionsCaptionBenchmark  +-- Flickr30kBenchmark
|   +-- HDTFBenchmark                       +-- InternVidBenchmark
|   +-- LAION400MBenchmark                  +-- LAION5BBenchmark
|   +-- MSCOCOCaptionBenchmark              `-- TextCapsBenchmark
+-- DetectionBenchmark
|   +-- Flickr30kEntitiesBenchmark          +-- MSCOCOBenchmark
|   |                                          `-- LVISBenchmark
|   `-- OpenImagesV4DetectionBenchmark
+-- MultipleChoiceBenchmark
|   +-- BLIP3o60kBenchmark      +-- ConceptualCaptionsBenchmark
|   `-- OpenVid1MBenchmark
+-- VideoClassificationBenchmark
|   +-- DFDCBenchmark           +-- KineticsBenchmark
|   `-- UCF101Benchmark
`-- VisualQABenchmark
    +-- DocVQABenchmark         +-- GQABenchmark
    +-- VisualCoTBenchmark      +-- VisualGenomeBenchmark
    `-- VQAv2Benchmark
```

## 4. `ModelRun` and `BenchmarkRun`

These are small immutable run-configuration objects, not additional model or benchmark base classes.

`ModelRun` records:

- `name`: the string used to identify the model in outputs.
- `model`: an instantiated object with `predict(...)`, such as a loaded `Qwen25VL3B` (`qwen2.5-vl-3b-instruct`) or `Llava` (`llava-1.5-7b-hf` by default).
- `load_time_seconds`: optional initialization time, automatically measured by `ModelRun.from_factory(...)`.

`BenchmarkRun` records:

- `benchmark`: an instantiated real benchmark such as `GQABenchmark()` or `MSCOCOCaptionBenchmark()`.
- `num_samples`: how many dataset rows this evaluation uses; values under one are rejected.

Creating a real `ModelRun.from_factory("qwen2.5-vl-3b-instruct", Qwen25VL3B, max_new_tokens=16)` would load the actual checkpoint, so this notebook displays its source contract rather than executing model construction.

In [ ]:
print(source_node("runners/model_run.py", "ModelRun"))
print("\n--- BenchmarkRun ---\n")
print(source_node("runners/benchmark_run.py", "BenchmarkRun"))

## 5. Executing a set of runs

`run_benchmark_matrix()` receives lists of the two run wrappers and nests two loops:

|  | `GQABenchmark` | `MSCOCOCaptionBenchmark` |
| --- | --- | --- |
| `qwen2.5-vl-3b-instruct` (`Qwen25VL3B`) | result JSON | result JSON |
| `llava-1.5-7b-hf` (`Llava`) | result JSON | result JSON |

In this example the executor would call `benchmark.run(model=..., n=...)` four times. The real runner stores the output from each call under `results/<model_name>_<benchmark_name>.json` and returns summaries containing those paths.

In [ ]:
print(source_node("runners/execution.py", "run_benchmark_matrix"))
print("\n--- A real repository test of the matrix behavior ---\n")
print(source_node("tests/test_benchmark_runner.py", "test_run_benchmark_matrix_runs_every_model_against_every_benchmark"))

## Extension guide

| Change wanted | Existing pattern to follow |
| --- | --- |
| New local vision-language model | Add a checkpoint leaf under an existing shared path where possible, such as `AutoImageTextToTextModelBase`, `LlavaNextModelBase`, or `_Qwen25VLBase`; implement new `predict(...)` logic only when the processor/model API differs. |
| New API-backed model | Reuse `OpenAIResponsesVisionModel` when it accepts Responses API image/text input, overriding only model-specific request behavior when required. |
| New dataset source | Implement `BaseDataset`, or reuse an `HF*Dataset` helper when the task matches an existing family. |
| New dataset for an existing task | Add a thin concrete benchmark specifying `dataset_cls`, like `GQABenchmark` or `MSCOCOCaptionBenchmark`. |
| New scoring task | Add a new `BaseBenchmark` subclass defining prompt and evaluation behavior, as the existing captioning or detection families do. |

The design rule visible throughout the repository is: datasets expose data and ground truth; benchmarks formulate and score tasks; models generate text; runners schedule combinations and persist reports.

## Source map

| Topic | File |
| --- | --- |
| Model contract | `models/_base_model.py` |
| Shared local-model loading | `models/_hf_model.py` |
| Local multimodal model example | `models/llava15_7b.py`, `models/qwen2_5_vl_3b_instruct.py` |
| Shared remote API inference | `models/_openai_vision.py`; thin checkpoint example: `models/gpt_4o.py` |
| Dataset contract and adapters | `dataset/_base_dataset.py`, `dataset/hf_common.py` |
| Concrete dataset examples | `dataset/gqa.py`, `dataset/mscoco_caption.py`, `dataset/imagenet1k.py` |
| Benchmark workflow | `benchmarks/_base_benchmark.py` |
| Benchmark families | `benchmarks/visual_qa/_visual_qa.py`, `benchmarks/captioning/_captioning.py`, `benchmarks/detection/_detection.py` |
| Run wrappers and executor | `runners/model_run.py`, `runners/benchmark_run.py`, `runners/execution.py` |
| Existing test stubs and runner tests | `tests/test_benchmark_runner.py` |